# PDelta2-Flash Layer Lab — Can the new layer beat Transformer attention?

This notebook tests a new **single-layer** TinyCeNN direction derived from the best previous result (`P-Delta2 F96`) and recent efficient-attention clues.

| Candidate ingredient | Why test it? |
|---|---|
| **P-Delta2 F96** | proven recurrent baseline |
| **Per-head output gate** | cheap selective read; inspired by gated linear/hybrid attention |
| **Causal Conv4 on V** | short local mixer used by recent linear-attention families |
| **Compact indexed block memory** | content-based long-range recall using one K/V summary per completed block |
| **Gate + Conv4 + Indexed** | strongest combined quality hypothesis |
| **F64 Gate + Conv4** | lean speed/memory candidate |
| **Exact trainable control** | exact Transformer softmax plus a tiny trainable head gain |

The benchmark uses document-disjoint train/validation/test partitions. Architecture selection is **validation-only**. The test set is opened only after winners are locked. A strict quality win requires the paired 95% bootstrap CI of `candidate NLL - Transformer NLL` to lie entirely below zero.


In [ ]:
import importlib, pathlib, subprocess, sys, tempfile

REF = 'main'
WORK = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
REPO = pathlib.Path(tempfile.mkdtemp(prefix='TinyCeNN-pdelta2-flash-', dir=WORK))
subprocess.run(['git','clone','--depth','1','--branch',REF,
                'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),
                'transformers==4.57.6','datasets>=3,<5','pandas','matplotlib','pytest>=8'], check=True)
for p in (REPO, REPO/'src'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
importlib.invalidate_caches()
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],
                                         text=True).strip())


## 1 · Choose the experiment budget

Start with **balanced**. Use `quick` only for pipeline checks; use `strong` after a promising winner appears.


In [ ]:
from datetime import datetime, timezone
import torch

PROFILE = 'balanced'  # quick | balanced | strong
LAYER = 18
TRAIN_CONTEXT = 256
TEST_CONTEXTS = '256,512,1024,2048'
SEED = 2026

assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = WORK / 'TinyCeNN-pdelta2-flash-results' / f'{PROFILE}-{RUN_ID}'
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Output:', OUTPUT_DIR)


## 2 · Fast correctness checks

These tests verify causality, function-preserving initialization, compact indexed memory, checkpoint reconstruction, and direct Colab-style CLI startup.


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',
                str(REPO/'tests/test_pdelta2_flash.py')], cwd=REPO, check=True)


## 3 · Train and evaluate the layer candidates

The child process is streamed into the notebook so a real Python traceback is visible immediately if anything fails.


In [ ]:
cmd = [
    sys.executable, str(REPO/'scripts/benchmark_pdelta2_flash_layer.py'),
    '--profile', PROFILE,
    '--layer', str(LAYER),
    '--train-context', str(TRAIN_CONTEXT),
    '--test-contexts', TEST_CONTEXTS,
    '--seed', str(SEED),
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='')
return_code = proc.wait()
if return_code:
    raise RuntimeError(f'PDelta2-Flash benchmark failed with exit code {return_code}')


## 4 · Validation ranking and locked selection


In [ ]:
import json, pandas as pd
from IPython.display import display

validation = pd.read_csv(OUTPUT_DIR/'validation_summary.csv')
tests = pd.read_csv(OUTPUT_DIR/'test_summary.csv')
selection = json.loads((OUTPUT_DIR/'selection.json').read_text())
report = json.loads((OUTPUT_DIR/'pdelta2_flash_report.json').read_text())

print(json.dumps(selection, indent=2))
cols = [
    'name','ingredient','validation_nll','validation_delta_nll','validation_ppl',
    'output_cosine','trainable_parameters','state_vs_transformer_fp16_256',
    'prefill_ms','decode_step_ms'
]
display(validation[[c for c in cols if c in validation.columns]])


## 5 · Did any candidate beat the Transformer?


In [ ]:
quality = tests[tests['candidate'] != 'transformer_original'].copy()
show = [
    'candidate','context','delta_nll','ci95_low','ci95_high','ppl_ratio',
    'state_vs_transformer_fp16','verdict','selected_quality','selected_efficient'
]
display(quality[[c for c in show if c in quality.columns]])

strict = quality[quality['verdict'] == 'strict_quality_win']
point = quality[quality['verdict'] == 'point_quality_win']
if len(strict):
    print('STRICT TRANSFORMER WINNER(S):')
    display(strict[show])
elif len(point):
    print('Point-estimate winner(s), but confidence interval still overlaps zero:')
    display(point[show])
else:
    print('No quality winner over Transformer in this run. Check memory-efficient parity and the Pareto plot below.')


## 6 · Quality / memory Pareto view


In [ ]:
import matplotlib.pyplot as plt

plot = quality[quality['context'] == TRAIN_CONTEXT].copy()
fig, ax = plt.subplots(figsize=(9,6))
ax.scatter(plot['state_vs_transformer_fp16'], plot['delta_nll'])
for _, row in plot.iterrows():
    ax.annotate(row['candidate'], (row['state_vs_transformer_fp16'], row['delta_nll']),
                xytext=(5,4), textcoords='offset points', fontsize=8)
ax.axhline(0, linewidth=1)
ax.axhline(0.02, linestyle='--', linewidth=1)
ax.set_xlabel('Persistent state / Transformer FP16 KV')
ax.set_ylabel('ΔNLL vs Transformer (lower is better)')
ax.set_title('PDelta2-Flash quality–memory Pareto')
ax.grid(True, alpha=0.25)
plt.show()

diag = report['winner_diagnostics']
print('Winner diagnostics:')
print(json.dumps(diag, indent=2))


## 7 · Download the complete result bundle

This final cell packages CSVs, JSON reports, checkpoints and all result artifacts into one ZIP and automatically downloads it.


In [ ]:
import shutil, pathlib
archive_base = OUTPUT_DIR.parent / OUTPUT_DIR.name
zip_path = pathlib.Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR))
print('ZIP:', zip_path, 'size:', zip_path.stat().st_size, 'bytes')

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('Automatic browser download is only available in Colab:', exc)
